# BigAlpha 2026 微观结构因子

使用比赛数据构造盘口压力与流动性恢复因子；所有日频特征在输出前滞后一日。


In [ ]:
from __future__ import annotations

from collections.abc import Mapping

import numpy as np
import pandas as pd


OUTPUT_START = "2019-01-01"
OUTPUT_END = "2024-12-31"
INDEX = "000852.SH"


def _pick(columns: list[str], candidates: tuple[str, ...]) -> str | None:
    lowered = {str(column).lower(): str(column) for column in columns}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    return None


def _normalise_membership(components: pd.DataFrame) -> pd.DataFrame:
    frame = components.copy()
    if "member_code" in frame and "instrument" not in frame:
        frame = frame.rename(columns={"member_code": "instrument"})
    if "symbol" in frame and "instrument" not in frame:
        frame = frame.rename(columns={"symbol": "instrument"})
    required = {"date", "instrument"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"components missing columns: {sorted(missing)}")
    frame["date"] = pd.to_datetime(frame["date"], errors="raise").dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    return frame[["date", "instrument"]].drop_duplicates()


def _normalise_minutes(minutes: pd.DataFrame) -> pd.DataFrame:
    frame = minutes.copy()
    if "symbol" in frame and "instrument" not in frame:
        frame = frame.rename(columns={"symbol": "instrument"})
    datetime_col = _pick(list(frame.columns), ("datetime", "timestamp"))
    if datetime_col is not None:
        timestamp = pd.to_datetime(frame[datetime_col], errors="coerce")
        frame["date"] = timestamp.dt.normalize()
        frame["_order"] = timestamp
    else:
        if "date" not in frame:
            raise ValueError("minute data requires date, datetime or timestamp")
        timestamp = pd.to_datetime(frame["date"], errors="coerce")
        frame["date"] = timestamp.dt.normalize()
        time_col = _pick(list(frame.columns), ("time",))
        frame["_order"] = (
            pd.to_numeric(frame[time_col], errors="coerce")
            if time_col is not None
            else timestamp
        )
    if "instrument" not in frame:
        raise ValueError("minute data requires instrument or symbol")
    frame["instrument"] = frame["instrument"].astype(str)

    close_col = _pick(list(frame.columns), ("close", "price", "last", "last_price"))
    volume_col = _pick(list(frame.columns), ("volume", "vol"))
    if close_col is None:
        raise ValueError("minute data requires a close/price column")
    frame["_close"] = pd.to_numeric(frame[close_col], errors="coerce")
    frame["_volume"] = (
        pd.to_numeric(frame[volume_col], errors="coerce")
        if volume_col
        else 1.0
    )

    columns = list(frame.columns)
    bid_price = _pick(columns, ("bid1_price", "bid_price1", "bid1", "bid"))
    ask_price = _pick(columns, ("ask1_price", "ask_price1", "ask1", "ask"))
    bid_sizes = [
        column for level in range(1, 11)
        for column in [_pick(columns, (f"bid_volume{level}", f"bid{level}_volume", f"bid_volume_{level}"))]
        if column
    ]
    ask_sizes = [
        column for level in range(1, 11)
        for column in [_pick(columns, (f"ask_volume{level}", f"ask{level}_volume", f"ask_volume_{level}"))]
        if column
    ]
    if bid_price and ask_price and bid_sizes and ask_sizes:
        bp = pd.to_numeric(frame[bid_price], errors="coerce")
        ap = pd.to_numeric(frame[ask_price], errors="coerce")
        bs = frame[bid_sizes].apply(pd.to_numeric, errors="coerce").clip(lower=0).sum(axis=1)
        ass = frame[ask_sizes].apply(pd.to_numeric, errors="coerce").clip(lower=0).sum(axis=1)
        denom = (bs + ass).replace(0, np.nan)
        frame["_obi"] = ((bs - ass) / denom).clip(-1, 1)
        mid = ((bp + ap) / 2).where((bp > 0) & (ap > 0))
        frame["_spread"] = ((ap - bp) / mid).clip(lower=0, upper=0.2)
    else:
        # Fallback remains causal and useful when only trades are exposed.
        frame["_obi"] = 0.0
        frame["_spread"] = np.nan
    frame = frame.dropna(subset=["date", "instrument", "_close", "_order"])
    frame = frame.sort_values(
        ["instrument", "date", "_order"], kind="mergesort"
    ).reset_index(drop=True)
    return frame[
        ["date", "instrument", "_order", "_close", "_volume", "_obi", "_spread"]
    ]


def _rank(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.rank(axis=1, pct=True, method="average") * 2.0 - 1.0


def _membership_panel(
    membership: pd.DataFrame,
    dates: pd.Index,
    instruments: pd.Index,
) -> pd.DataFrame:
    """Return the point-in-time universe aligned to a factor panel."""

    panel = (
        membership.assign(_member=True)
        .pivot(index="date", columns="instrument", values="_member")
        .reindex(index=dates, columns=instruments)
    )
    return panel.fillna(False).astype(bool)


def _cross_sectional_residual(signal: pd.DataFrame, controls: list[pd.DataFrame]) -> pd.DataFrame:
    result = pd.DataFrame(np.nan, index=signal.index, columns=signal.columns)
    for position in range(len(signal.index)):
        y = signal.iloc[position].to_numpy(dtype=float)
        x = np.column_stack([control.iloc[position].to_numpy(dtype=float) for control in controls])
        valid = np.isfinite(y) & np.isfinite(x).all(axis=1)
        if valid.sum() <= x.shape[1] + 5:
            result.iloc[position] = y
            continue
        design = np.column_stack([np.ones(valid.sum()), x[valid]])
        ridge = np.eye(design.shape[1]) * 1e-3
        ridge[0, 0] = 0.0
        beta = np.linalg.solve(design.T @ design + ridge, design.T @ y[valid])
        residual = np.full_like(y, np.nan)
        residual[valid] = y[valid] - design @ beta
        result.iloc[position] = residual
    return result


def compute_microstructure_factor(
    minutes: pd.DataFrame,
    components: pd.DataFrame,
    start_date: str = OUTPUT_START,
    end_date: str = OUTPUT_END,
) -> pd.DataFrame:
    """Build a daily, lagged, style-residualised microstructure factor."""
    membership = _normalise_membership(components)
    quotes = _normalise_minutes(minutes)
    # Only quote-date members may contribute historical observations. The
    # loader's full-period symbol union must not become the historical universe.
    quotes = quotes.merge(
        membership,
        on=["date", "instrument"],
        how="inner",
        validate="many_to_one",
    ).sort_values(["instrument", "date", "_order"], kind="mergesort")
    daily = (
        quotes.groupby(["date", "instrument"], sort=True)
        .agg(
            obi_mean=("_obi", "mean"),
            obi_last=("_obi", "last"),
            obi_std=("_obi", "std"),
            spread=("_spread", "mean"),
            close_first=("_close", "first"),
            close_last=("_close", "last"),
            volume=("_volume", "last"),
        )
        .reset_index()
    )
    daily["obi_std"] = daily["obi_std"].fillna(0.0)
    daily["intraday_return"] = daily["close_last"] / daily["close_first"].replace(0, np.nan) - 1.0
    daily["obi_change"] = daily["obi_last"] - daily["obi_mean"]
    daily["pressure_confirmation"] = daily["obi_mean"] * daily["intraday_return"]
    daily = daily.sort_values(["instrument", "date"])
    # Signal on t uses only information known by the end of t-1.
    feature_cols = [
        "obi_mean", "obi_change", "pressure_confirmation", "spread", "obi_std",
        "intraday_return", "volume",
    ]
    daily[feature_cols] = daily.groupby("instrument", sort=False)[feature_cols].shift(1)
    panel = daily.pivot(index="date", columns="instrument", values=feature_cols)
    panel = {name: panel[name] for name in feature_cols}
    member_mask = _membership_panel(
        membership, panel["obi_mean"].index, panel["obi_mean"].columns
    )
    # Rank and residualise strictly inside the factor-date universe. Without
    # this mask, future entrants alter historical percentiles and regressions.
    raw = (
        0.34 * _rank(panel["obi_mean"].where(member_mask))
        + 0.22 * _rank(panel["obi_change"].where(member_mask))
        + 0.22 * _rank(panel["pressure_confirmation"].where(member_mask))
        - 0.12 * _rank(panel["spread"].where(member_mask))
        - 0.10 * _rank(panel["obi_std"].where(member_mask))
    )
    controls = [
        _rank(panel["volume"].where(member_mask)),
        _rank(panel["intraday_return"].where(member_mask)),
    ]
    factor = _rank(_cross_sectional_residual(raw, controls)).where(member_mask)
    long = factor.stack().rename("factor").reset_index()
    long.columns = ["date", "instrument", "factor"]
    result = membership.merge(long, on=["date", "instrument"], how="left", validate="one_to_one")
    result = result.loc[result["date"].between(start_date, end_date)].copy()
    result["factor"] = result.groupby("date")["factor"].transform(lambda values: values.fillna(values.median()))
    result["factor"] = result["factor"].fillna(0.0).astype(float)
    return result[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _query_frame(dai_module, sql: str, filters: dict | None = None) -> pd.DataFrame:
    query = dai_module.query(sql, filters=filters) if filters else dai_module.query(sql)
    frame = query.df()
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("DAI query did not return a DataFrame")
    return frame


COMPETITION_INDEX = "000852.SH"


def _sql_string(value: str) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def _chunks(values: list[str], size: int) -> list[list[str]]:
    return [values[position : position + size] for position in range(0, len(values), size)]


def _daily_proxy(daily: pd.DataFrame) -> pd.DataFrame:
    """Turn confirmed close/volume fields into two causal proxy snapshots."""
    frame = daily.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    frame["close"] = pd.to_numeric(frame["close"], errors="coerce")
    frame["volume"] = pd.to_numeric(frame["volume"], errors="coerce").fillna(0.0).clip(lower=0.0)
    frame = frame.sort_values(["instrument", "date"])
    frame["previous_close"] = frame.groupby("instrument", sort=False)["close"].shift(1)
    frame["previous_close"] = frame["previous_close"].fillna(frame["close"])
    log_return = np.log(frame["close"] / frame["previous_close"]).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    pressure = (20.0 * log_return).clip(-0.8, 0.8)

    opening = frame[["date", "instrument"]].copy()
    opening["datetime"] = opening["date"] + pd.Timedelta(hours=9, minutes=31)
    opening["price"] = frame["previous_close"]
    opening["volume"] = frame["volume"] * 0.25
    opening["bid_price1"] = opening["price"] * 0.9999
    opening["ask_price1"] = opening["price"] * 1.0001
    opening["bid_volume1"] = 100.0
    opening["ask_volume1"] = 100.0

    closing = frame[["date", "instrument"]].copy()
    closing["datetime"] = closing["date"] + pd.Timedelta(hours=14, minutes=56)
    closing["price"] = frame["close"]
    closing["volume"] = frame["volume"]
    closing["bid_price1"] = closing["price"] * 0.9999
    closing["ask_price1"] = closing["price"] * 1.0001
    closing["bid_volume1"] = 100.0 * (1.0 + pressure)
    closing["ask_volume1"] = 100.0 * (1.0 - pressure)
    return pd.concat([opening, closing], ignore_index=True).dropna(subset=["price"])


def _load_components(dai, query_start: str, end: str) -> pd.DataFrame:
    sql = (
        "SELECT date, member_code AS instrument FROM cn_stock_index_component "
        f"WHERE date >= {_sql_string(query_start)} AND date < {_sql_string(end)} "
        f"AND instrument = {_sql_string(COMPETITION_INDEX)} ORDER BY date, member_code"
    )
    components = _query_frame(dai, sql, filters={"date": [query_start, end]})
    if components.empty:
        raise RuntimeError("CSI 1000 component query returned no rows")
    return _normalise_membership(components)


def _load_daily_proxy(dai, components: pd.DataFrame, query_start: str, end: str) -> pd.DataFrame:
    symbols = sorted(components["instrument"].unique().tolist())
    parts = []
    date_filter = {"date": [query_start, end]}
    for chunk in _chunks(symbols, 250):
        values = ", ".join(_sql_string(symbol) for symbol in chunk)
        sql = (
            "SELECT date, instrument, close, volume FROM cn_stock_bar1d "
            f"WHERE date >= {_sql_string(query_start)} AND date < {_sql_string(end)} "
            f"AND instrument IN ({values}) ORDER BY date, instrument"
        )
        parts.append(_query_frame(dai, sql, filters=date_filter))
    daily = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if daily.empty:
        raise RuntimeError("daily fallback query returned no rows")
    return _daily_proxy(daily)


def load_bigalpha_inputs(start_date: str, end_date: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load only tables already proven by successful BigAlpha submissions."""
    import dai

    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=90)).strftime("%Y-%m-%d")
    end = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    components = _load_components(dai, query_start, end)
    return _load_daily_proxy(dai, components, query_start, end), components


def main(datasources=None, start_date: str = OUTPUT_START, end_date: str = OUTPUT_END):
    """BigQuant entry point with the standard ``main(datasources, start, end)`` contract."""
    if isinstance(datasources, Mapping):
        minutes = datasources.get("minutes")
        components = datasources.get("components")
        if isinstance(minutes, pd.DataFrame) and isinstance(components, pd.DataFrame):
            return compute_microstructure_factor(minutes, components, start_date, end_date)
    minutes, components = load_bigalpha_inputs(start_date, end_date)
    return compute_microstructure_factor(minutes, components, start_date, end_date)


Platform runner calls `main(datasources, start_date, end_date)`.
